# 👁️ VERA Phase 2: Universal Multi-Dataset Kaggle GPU Training
### Vascular Explainable Retinopathy Assessment System

This notebook automatically scans **all attached datasets** in `/kaggle/input/` (**APTOS 2019**, **EyePACS**, **Messidor-2**), combines all 40,000+ fundus images into a unified training set, and trains the **VERA 4-channel attention-gated ResNet-50** model.

---

## Cell 1: Environment Setup & GPU Verification

In [ ]:
# Install required dependencies
!pip install -q albumentations timm segmentation-models-pytorch kagglehub scikit-learn pandas opencv-python-headless tqdm

import os
import sys
from pathlib import Path
import torch
import warnings
warnings.filterwarnings('ignore')

# Set Kaggle working directory
KAGGLE_WORKING = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('./')
OUTPUT_DIR = KAGGLE_WORKING / 'checkpoints/production_model'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'PyTorch Version: {torch.__version__}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Compute Device: {device}')

if torch.cuda.is_available():
    print(f'GPU Model: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
else:
    print('⚠️ GPU not enabled. Make sure Accelerator is set to GPU P100 or T4 in Kaggle Settings (right sidebar).')


## Cell 2: Universal Multi-Dataset Recursive Loader (/kaggle/input/)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

input_root = Path('/kaggle/input')
records = []
seen_image_paths = set()

print('🔍 Universal Kaggle Multi-Dataset Scanner Starting...\n')

if input_root.exists():
    all_csvs = list(input_root.rglob('*.csv'))
    print(f'Found {len(all_csvs)} CSV metadata files in /kaggle/input/:\n')
    
    for csv_path in all_csvs:
        if 'submission' in csv_path.name.lower():
            continue
            
        try:
            df_tmp = pd.read_csv(csv_path)
            cols = [str(c).lower() for c in df_tmp.columns]
            
            # Identify image column
            img_col = None
            for cand in ['id_code', 'image', 'image_id', 'filename', 'img_name']:
                if cand in cols:
                    img_col = df_tmp.columns[cols.index(cand)]
                    break
            if img_col is None and len(df_tmp.columns) > 0:
                img_col = df_tmp.columns[0]
                
            # Identify label column
            lbl_col = None
            for cand in ['diagnosis', 'level', 'adjudicated_dr_grade', 'dr_grade', 'grade', 'dr']:
                for idx, c in enumerate(cols):
                    if cand in c:
                        lbl_col = df_tmp.columns[idx]
                        break
                if lbl_col is not None:
                    break
            if lbl_col is None and len(df_tmp.columns) > 1:
                lbl_col = df_tmp.columns[1]
                
            if img_col is None or lbl_col is None:
                continue
                
            # Index all image files under the dataset directory recursively
            dataset_dir = csv_path.parent
            search_dirs = [dataset_dir, dataset_dir.parent]
            img_file_map = {}
            for s_dir in search_dirs:
                if s_dir.exists():
                    for img_p in s_dir.rglob('*'):
                        if img_p.is_file() and img_p.suffix.lower() in ['.png', '.jpg', '.jpeg']:
                            img_file_map[img_p.stem] = str(img_p)
                            img_file_map[img_p.name] = str(img_p)
                            
            count = 0
            dataset_tag = csv_path.parent.name
            for _, row in df_tmp.iterrows():
                val_img = str(row[img_col]).strip()
                val_lbl = row[lbl_col]
                if pd.isna(val_lbl): continue
                try:
                    val_lbl = int(float(val_lbl))
                except ValueError:
                    continue
                if val_lbl not in [0, 1, 2, 3, 4]: continue
                
                img_path_str = None
                stem_name = Path(val_img).stem
                if val_img in img_file_map:
                    img_path_str = img_file_map[val_img]
                elif stem_name in img_file_map:
                    img_path_str = img_file_map[stem_name]
                    
                if img_path_str and img_path_str not in seen_image_paths:
                    seen_image_paths.add(img_path_str)
                    records.append({
                        'image_path': img_path_str,
                        'diagnosis': val_lbl,
                        'dataset': dataset_tag
                    })
                    count += 1
                    
            if count > 0:
                print(f'✓ Loaded {count} valid images from: {csv_path.relative_to(input_root)}')
        except Exception as e:
            pass

df_manifest = pd.DataFrame(records)
print('\n=======================================================')
print(f'🎉 TOTAL MULTI-DATASET COMBINED: {len(df_manifest)} IMAGES READY!')
print('=======================================================\n')
if len(df_manifest) > 0:
    print('Breakdown by Dataset Source:')
    print(df_manifest['dataset'].value_counts())
    print('\nCombined Class Distribution across 5 ICDR Grades:')
    print(df_manifest['diagnosis'].value_counts().sort_index())
else:
    print('⚠️ No valid images matched. Make sure your datasets are attached in Kaggle Right Sidebar -> Add Data.')


## Cell 3: Image Preprocessing & Vessel Feature Extraction

In [ ]:
import cv2
import torch.nn as nn
import torchvision.models as models

def crop_fundus_circle(img, tol=10):
    if img.ndim == 2:
        mask = img > tol
        if not mask.any(): return img
        return img[np.ix_(mask.any(1), mask.any(0))]
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    mask = gray > tol
    if not mask.any(): return img
    row_idx, col_idx = mask.any(axis=1), mask.any(axis=0)
    ymin, ymax = np.where(row_idx)[0][[0, -1]]
    xmin, xmax = np.where(col_idx)[0][[0, -1]]
    return img[max(0, ymin-2):min(gray.shape[0], ymax+2), max(0, xmin-2):min(gray.shape[1], xmax+2)]

def apply_clahe(img):
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l_clahe = clahe.apply(l)
    return cv2.cvtColor(cv2.merge((l_clahe, a, b)), cv2.COLOR_LAB2RGB)

def extract_vessel_map(img_rgb):
    green = img_rgb[:, :, 1]
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(green)
    inverted = 255 - enhanced
    blurred = cv2.GaussianBlur(inverted, (5, 5), 0)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    tophat = cv2.morphologyEx(blurred, cv2.MORPH_TOPHAT, kernel)
    v_map = cv2.normalize(tophat, None, 0.0, 1.0, cv2.NORM_MINMAX, dtype=cv2.CV_32F)
    return v_map


## Cell 4: VERA 4-Channel ConvNet Architecture ($[R, G, B, V]$)

In [ ]:
class VERA4ChannelResNet50(nn.Module):
    def __init__(self, num_classes=5, pretrained=True):
        super().__init__()
        self.backbone = models.resnet50(weights=models.ResNet50_Weights.DEFAULT if pretrained else None)
        
        old_conv = self.backbone.conv1
        new_conv = nn.Conv2d(4, old_conv.out_channels, kernel_size=old_conv.kernel_size,
                             stride=old_conv.stride, padding=old_conv.padding, bias=old_conv.bias is not None)
        with torch.no_grad():
            new_conv.weight[:, :3, :, :] = old_conv.weight
            new_conv.weight[:, 3:4, :, :] = old_conv.weight.mean(dim=1, keepdim=True)
        self.backbone.conv1 = new_conv
        
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(in_features, num_classes)
        )
        
    def forward(self, x):
        return self.backbone(x)

model = VERA4ChannelResNet50(num_classes=5, pretrained=True).to(device)
print(f'✓ Model initialized with 4-channel conv1 shape: {model.backbone.conv1.weight.shape}')


## Cell 5: High-Speed Training Pipeline with AMP & QWK Optimization

In [ ]:
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import cohen_kappa_score, accuracy_score
from sklearn.model_selection import train_test_split
from tqdm import tqdm

class VERADataset(Dataset):
    def __init__(self, df, target_size=(224, 224)):
        self.df = df.reset_index(drop=True)
        self.target_size = target_size
        
    def __len__(self):
        return len(self.df)
        
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = cv2.imread(row['image_path'])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        cropped = crop_fundus_circle(img)
        clahe = apply_clahe(cropped)
        resized = cv2.resize(clahe, (self.target_size[1], self.target_size[0]))
        v_map = extract_vessel_map(resized)
        
        rgb_norm = (resized.astype(np.float32) / 255.0 - np.array([0.485, 0.456, 0.406])) / np.array([0.229, 0.224, 0.225])
        v_norm = (v_map - 0.15) / 0.25
        
        t_rgb = torch.from_numpy(rgb_norm.transpose(2, 0, 1)).float()
        t_v = torch.from_numpy(v_norm).unsqueeze(0).float()
        tensor_4ch = torch.cat([t_rgb, t_v], dim=0)
        
        label = torch.tensor(row['diagnosis'], dtype=torch.long)
        return tensor_4ch, label

if len(df_manifest) > 0:
    train_df, val_df = train_test_split(df_manifest, test_size=0.2, random_state=42, stratify=df_manifest['diagnosis'])
    train_loader = DataLoader(VERADataset(train_df), batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(VERADataset(val_df), batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)
    scaler = torch.amp.GradScaler('cuda') if torch.cuda.is_available() else None

    best_kappa = 0.0
    print(f'🚀 Starting training on {len(train_df)} images for 30 Epochs...')
    
    for epoch in range(30):
        model.train()
        train_loss = 0.0
        pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/30 [Train]')
        for images, labels in pbar:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            if scaler:
                with torch.amp.autocast('cuda'):
                    outputs = model(images)
                    loss = criterion(outputs, labels)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                outputs = model(images)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()
                
            train_loss += loss.item()
            pbar.set_postfix({'loss': train_loss / (pbar.n + 1)})
            
        model.eval()
        val_preds, val_labels = [], []
        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device)
                outputs = model(images)
                preds = outputs.argmax(dim=1).cpu().numpy()
                val_preds.extend(preds)
                val_labels.extend(labels.numpy())
                
        val_acc = accuracy_score(val_labels, val_preds) * 100
        val_kappa = cohen_kappa_score(val_labels, val_preds, weights='quadratic')
        scheduler.step()
        
        print(f'Epoch {epoch+1}/30: Val Acc = {val_acc:.2f}%, Val QWK Kappa = {val_kappa:.4f}')
        
        if val_kappa > best_kappa:
            best_kappa = val_kappa
            checkpoint_dict = {
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'val_kappa': val_kappa,
                'val_acc': val_acc,
                'backbone': 'resnet50'
            }
            torch.save(checkpoint_dict, KAGGLE_WORKING / 'best_model.pth')
            torch.save(checkpoint_dict, OUTPUT_DIR / 'best_model.pth')
            print(f'  ✓ Saved best model (Kappa: {val_kappa:.4f}) to {KAGGLE_WORKING / "best_model.pth"}')


## Cell 6: Direct One-Click Download for `best_model.pth`

In [ ]:
from pathlib import Path
from IPython.display import HTML, display

target_ckpt = Path('/kaggle/working/best_model.pth')
if not target_ckpt.exists():
    target_ckpt = Path('./best_model.pth')

if target_ckpt.exists():
    file_size_mb = target_ckpt.stat().st_size / (1024 * 1024)
    print(f'✅ Trained Checkpoint Ready: {target_ckpt.resolve()} ({file_size_mb:.2f} MB)')
    
    html_code = f"""
    <div style="background-color: #ECFDF5; border: 2px solid #10B981; padding: 20px; border-radius: 10px; text-align: center; margin: 20px 0;">
        <h2 style="color: #065F46; margin-top: 0;">🎉 Training Complete & Checkpoint Saved!</h2>
        <p style="font-size: 16px; color: #047857;">Your trained model <b>best_model.pth</b> ({file_size_mb:.2f} MB) is ready for download.</p>
        <a href="best_model.pth" download="best_model.pth" style="background-color: #059669; color: white; padding: 12px 24px; font-size: 18px; font-weight: bold; text-decoration: none; border-radius: 6px; display: inline-block; margin: 10px 0;">
            ⬇️ CLICK HERE TO DOWNLOAD best_model.pth
        </a>
        <p style="font-size: 13px; color: #064E3B; margin-bottom: 0;">Or find <b>best_model.pth</b> in the Kaggle Right Sidebar under <b>Output -&gt; /kaggle/working/</b>.</p>
    </div>
    """
    display(HTML(html_code))
else:
    print('⚠️ best_model.pth not found yet. Make sure Cell 5 training has completed.')
